In [0]:
%sql
-- Load DIM_DEPARTMENT
-- ----------------------------------------------------------------------------
INSERT INTO university_bot.analytics.dim_department
    (department_id, department_name, faculty_id, created_at, updated_at)
SELECT DISTINCT
    department_id,
    department_name,
    faculty_id,
    created_at,
    updated_at
FROM university_bot.core.departments;

-- ----------------------------------------------------------------------------
-- Load DIM_PROFESSOR
-- ----------------------------------------------------------------------------
INSERT INTO university_bot.analytics.dim_professor
    (professor_id, professor_name, professor_email, academic_title,
     department_id, created_at, updated_at)
SELECT DISTINCT
    professor_id,
    professor_name,
    professor_email,
    academic_title,
    department_id,
    created_at,
    updated_at
FROM university_bot.core.professors;

-- ----------------------------------------------------------------------------
-- Load DIM_COURSE
-- ----------------------------------------------------------------------------
INSERT INTO university_bot.analytics.dim_course
    (course_id, course_code, course_name, credit_hours, study_year,
     semester, department_id, created_at, updated_at)
SELECT DISTINCT
    course_id,
    course_code,
    course_name,
    credit_hours,
    study_year,
    semester,
    department_id,
    created_at,
    updated_at
FROM university_bot.core.courses;

-- ----------------------------------------------------------------------------
-- Load DIM_STUDENT
-- ----------------------------------------------------------------------------
INSERT INTO university_bot.analytics.dim_student
    (student_id, national_id, student_name, phone, gender, birthdate,
     study_year, department_id, enrollment_year, telegram_username,
     created_at, updated_at)
SELECT DISTINCT
    student_id,
    national_id,
    student_name,
    phone,
    gender,
    birthdate,
    study_year,
    department_id,
    enrollment_year,
    telegram_username,
    created_at,
    updated_at
FROM university_bot.core.students;

-- ----------------------------------------------------------------------------
-- Load DIM_SESSION
-- ----------------------------------------------------------------------------
INSERT INTO university_bot.analytics.dim_session
    (session_id, course_id, professor_id, session_type, day_of_week,
     start_time, end_time, location, semester, academic_year,
     created_at, updated_at)
SELECT DISTINCT
    session_id,
    course_id,
    professor_id,
    session_type,
    day_of_week,
    start_time,
    end_time,
    location,
    semester,
    academic_year,
    created_at,
    updated_at
FROM university_bot.core.sessions;


/* ============================================================================
   STEP 6: POPULATE FACT_ATTENDANCE
   ----------------------------------------------------------------------------
   The OLTP 'attendance' table only carries student_id and session_id.
   course_id and professor_id are not stored directly on attendance; they
   are derived by joining through the session (a session belongs to exactly
   one course and one professor). The join chain is:

       attendance --(session_id)--> sessions --(course_id)--> courses
                                             --(professor_id)--> professors

   Each natural key is then resolved to its corresponding dimension
   surrogate key. DISTINCT protects against fan-out duplicates in case of
   any repeated rows in the source attendance table.
   ============================================================================ */
INSERT INTO university_bot.analytics.fact_attendance
    (student_key, course_key, professor_key, session_key,
     attendance_date, attendance_status, check_in_time, created_at)
SELECT DISTINCT
    ds.student_key,
    dc.course_key,
    dp.professor_key,
    dse.session_key,
    a.attendance_date,
    a.status              AS attendance_status,
    a.check_in_time,
    a.created_at
FROM university_bot.analytics.dim_student AS a
-- Resolve session first, since course_id / professor_id are derived from it
INNER JOIN university_bot.analytics.dim_session dse
    ON a.session_id = dse.session_id
-- Resolve student surrogate key
INNER JOIN university_bot.analytics.dim_student ds
    ON a.student_id = ds.student_id
-- Resolve course surrogate key via the session's course_id
INNER JOIN university_bot.analytics.dim_course dc
    ON dse.course_id = dc.course_id
-- Resolve professor surrogate key via the session's professor_id
INNER JOIN university_bot.analytics.dim_professor dp
    ON dse.professor_id = dp.professor_id;
